"""
This code is provided as supplemental material to the publication "Machine learning for small data sets: an exemplary study on the classification of highly complex surface micromorphologies" 
by M. Henkel, M. Sprenger, and O. Lieleg submitted to Materials Today Advances on October 17th, 2025.

"""

In [ ]:
#Import Data
import sklearn as skl
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, MinMaxScaler, QuantileTransformer, PowerTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn import decomposition
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from NETCORE import *

In [ ]:
def clustering():
    data, labels = import_data()
    features = run_NETCORE(data, t_corr = 0.8) 
    data = data.loc[:,features]
    data = preprocessing(data, features)
    data, columns= PCA(data, labels, features)
    data, silhouette_scores, n_clusters = k_means(data, columns)
    plot_silhouette(silhouette_scores)
    plot_output(data,cluster_col='Class_2', colors = 'Blues')
    plot_output(data,cluster_col='Cluster_3', colors = 'Blues')
    plot_output(data,cluster_col='Class_1', colors = 'Reds')
    plot_output(data,cluster_col='Cluster_6', colors = 'Reds')


In [ ]:
def import_data():
    # Specify the paths 
    csv_file_path = ['Data/ABR_ADH_surface_param.csv','Data/ABR_ERO_surface_param.csv', 'Data/ADH_ABR_surface_param.csv', 'Data/ADH_ERO_surface_param.csv', 'Data/ERO_ABR_surface_param.csv', 'Data/ERO_ADH_surface_param.csv']         
    data = []
    for filename in csv_file_path:
        data.append(pd.read_csv(filename,  delimiter =';', decimal = ',', engine ='python'))
    data = pd.concat(data, ignore_index=True)
    data = data.dropna()
    labels = data.loc[:,['Class_1', 'Class_2']]                                                        
    data = data.drop(['Area size', 'File name', 'Class_1','Class_2'], axis=1)

    return data, labels

In [ ]:
def preprocessing(data,features):
    scalers = [
                QuantileTransformer( output_distribution = 'uniform'),  
                MinMaxScaler(),  
            ]
    for scaler in scalers:
        data = scaler.fit_transform(data)
    data = pd.DataFrame(data, columns=features)
    return data

In [ ]:
def k_means(data, columns):
    start_index = 2
    
    n_clusters_range = list(range(start_index, 9))
    silhouette_scores_list = []
    davies_bouldin_scores_list = []
    clustering_results = []

    # K-Means Clustering for different number of clusters
    for n_clusters in n_clusters_range:
        kmeans = KMeans(
                n_clusters=n_clusters,      
                init='k-means++',         
                max_iter=300,             
                n_init=10,                 
                random_state=4 
                )
        cluster_labels = kmeans.fit_predict(data[columns])
        clustering_results.append(cluster_labels)
        data[f'Cluster_{n_clusters}'] = cluster_labels
        
        #Silhoutte Scores
        silhouette_scores_list.append(silhouette_score(data[columns], cluster_labels))
        davies_bouldin_scores_list.append(davies_bouldin_score(data[columns], cluster_labels))

    pd_silhouette_scores = pd.DataFrame({
        'Number of Clusters': n_clusters_range,
        'Silhouette - Score': silhouette_scores_list,
        'David Bouldin Score': davies_bouldin_scores_list
        })  
         
    # Find the best silhouette score and corresponding index
    best_score = pd_silhouette_scores['Silhouette - Score'].max()
    index = pd_silhouette_scores['Silhouette - Score'].idxmax()

    # Get the corresponding best number of clusters
    best_n_clusters = n_clusters_range[index]            
        
    print('\n Best value for n_clusters according to Silhoutte-Score (%f) is : %i with %i unique clusters identified' %(best_score, best_n_clusters, len(set(clustering_results[index]))))
    print(pd_silhouette_scores)
    return data, pd_silhouette_scores, n_clusters

In [ ]:
def PCA(data,labels, features):
    # Perform PCA analysis
    dim = 5
    pca = decomposition.PCA(n_components=dim, copy=True)
    surface_data = data.loc[:, features]  
    pca_results = pca.fit_transform(surface_data)

    # Explained variance
    explained = pca.explained_variance_ratio_
    print('\nExplained variance by PCA components:', explained)
    total_explained = explained.sum()
    print('\nTotal explained variance by PCA components:', total_explained)

    # Create DataFrame with PCA results
    columns=['PC1', 'PC2', 'PC3', 'PC4', 'PC5']
    dim_reduced_data = pd.DataFrame(pca_results, columns=columns)
    dim_reduced_data.insert(0, 'Sample', [f'Sample_{i+1}' for i in range(len(data))])
    dim_reduced_data.insert(1, 'Class_1', labels['Class_1'].values)
    dim_reduced_data.insert(1, 'Class_2', labels['Class_2'].values)
    for col in data.columns.difference(features):
        dim_reduced_data.insert(1, col, data[col].values)

    return dim_reduced_data, columns

In [ ]:
def plot_silhouette(silhouette_scores):
    plt.figure(figsize=(40, 6), dpi=300)
    plt.plot(silhouette_scores['Number of Clusters'], 
            silhouette_scores['Silhouette - Score'],
            marker='o', 
            markersize=8,
            linewidth=2,
            label='Silhouette Score', 
            color='green')
    plt.xticks(silhouette_scores['Number of Clusters'])
    plt.xlabel('Number of Clusters', fontsize=12)
    plt.ylabel('Silhouette Score', fontsize=12)
    plt.title('Silhouette Scores for Different Number of Clusters', fontsize=14)
    plt.tight_layout()
    plt.savefig('Silhouette_SCORE.eps', 
                format='eps', 
                dpi=300, 
                bbox_inches='tight',
                pad_inches=0.1)
    plt.show()
    plt.close()

In [ ]:
def plot_output(data, cluster_col, colors):
    if cluster_col not in data.columns:
        print(f"Input '{cluster_col}' not found!")
        return
    labels, uniques = pd.factorize(data[cluster_col])
    n_colors = labels.max() + 1
    base_cmap = plt.get_cmap(colors)
    cmap = ListedColormap(base_cmap(np.linspace(0.2, 1, n_colors)))

    plt.scatter(data['PC1'], data['PC2'], c=labels, cmap=cmap, s=40)
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.title(f'{cluster_col}')
    cbar = plt.colorbar(label='Cluster') 
    cbar.set_ticks(np.arange(0, n_colors))
    cbar.set_ticklabels(uniques)
    plt.savefig(f'{cluster_col}.eps', format='eps', bbox_inches='tight')
    plt.show()
    plt.close()


    

In [ ]:
clustering()